Basado en el seminario de verano ISIE Bath - 2026
Creditos a: Simon Schulte, Johan Perez, Stefan Pauliuk


## Manejo básico de datos: leer e inspeccionar datos, graficar y guardar resultados

Este cuaderno de Jupyter contiene explicaciones sobre cómo usar el lenguaje de programación Python para leer, inspeccionar y graficar datos, así como para guardar los resultados de los cálculos.

Los pasos de manejo de datos y de cálculo se explican paso a paso. 
Se puede obtener información adicional sobre cómo trabajar con los cuadernos en 
https://www.dataquest.io/blog/jupyter-notebook-tutorial/

Primero: este recuadro es un cuadro de texto en el llamado formato markdown, que permite introducir algunas instrucciones simples de formato, como el título de arriba (con numerales), o texto en negrita o cursiva:

**Hola, esto es un texto**

_Estilo cursiva_

Aquí podemos explicar y documentar cosas.

## Planteamiento del problema: monitorear el nexo stock-flujo-servicio para vehículos de pasajeros en la UE

Los vehículos de pasajeros son grandes consumidores de energía y materiales, y fuentes importantes de GEI. Una transformación de la flota de vehículos de pasajeros hacia autos eléctricos de batería, combinada con electricidad baja en carbono, puede reducir el impacto del nexo en los GEI, pero tiene implicancias en los ciclos de materiales.

**Durante las próximas semanas aprenderemos a modelar el nexo stock-flujo-servicio en Python, enfocándonos en los vehículos de pasajeros de los 20 países de la zona euro como caso de estudio.** (Este alcance geográfico se eligió por la disponibilidad de datos.)

**Este primer cuaderno de software introduce los pasos básicos: leer e inspeccionar datos, graficar y guardar resultados.**

El libro de datos *Bath_dMFA_2026_Software_Data.xlsx* contiene los parámetros relevantes para el cálculo.

En un primer paso, leemos algunos de estos datos en Python, los inspeccionamos, los graficamos, realizamos algunos cálculos básicos y guardamos los resultados.

In [ ]:
# importar las librerías necesarias
import pandas as pd   # librería de dataframes, usada para importar/exportar rápidamente desde/hacia Excel y para organizar los datos.
import numpy as np    # librería matemática
import matplotlib.pyplot as plt  # librería para gráficos

In [ ]:
# leer los datos de entrada desde Excel a un dataframe
df_I = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='New registration', index_col=0)
df_I # mostrar los datos

### Tarea 1: ¿Qué significan los datos mostrados arriba? ¿Qué significa el índice de las filas y cuál es la unidad de los datos? ¿Cuál fue el flujo de entrada (matriculación de autos nuevos) en 2000, 2010 y 2020? ¿Qué tendencia se observa en este conjunto de datos?

Las primeras preguntas pueden responderse directamente con la información mostrada arriba.
Para abordar la última pregunta, graficamos la serie temporal de la tabla anterior para hacer una buena inspección visual.
El código de Python de abajo lo hace.

In [ ]:
plt.plot(np.arange(1990,2025,1), df_I.values) # crear un gráfico de líneas simple con la serie temporal
plt.ylim(0, 1.1*np.max(df_I.values))
plt.xlabel('cohorte de edad') # agregar etiqueta del eje x
plt.ylabel('matriculaciones nuevas (entrada), autos') # agregar etiqueta del eje y
plt.title('Matriculación de autos de pasajeros por cohorte de edad, 20 países de la zona euro') # agregar título
plt.show(); # mostrar el gráfico con todas las características anteriores

Con la información anterior, ya se puede responder la pregunta restante:

**La matriculación de vehículos de pasajeros en los países de la zona euro se ha estabilizado en el rango de 10-11 millones de unidades por año, con una tendencia ligeramente decreciente.**

Ahora inspeccionaremos algunos de los fragmentos de código anteriores para entender qué hacen:

In [ ]:
np.arange(1990,2025,1)

In [ ]:
np.arange(1996,2017,3)

El comando np.arange simplemente genera una serie de números naturales, usando los valores inicial y final dados y el espaciado indicado. Nótese que el valor final indicado nunca forma parte del resultado; para obtener una serie de años hasta 2060, hay que indicar 2061 como límite:

In [ ]:
np.arange(2025,2061,1)

In [ ]:
df_I.values

Este comando devuelve los valores del dataframe (el contenedor que usamos para leer los datos desde Excel). Nótese que las etiquetas de las filas (los años) no forman parte de los datos.

Mediante la indexación se puede acceder a valores individuales y a rangos de valores, una técnica llamada _slicing_ (rebanado).

¡Nótese que la indexación siempre comienza en 0 y termina en el penúltimo valor indicado!

In [ ]:
df_I.values[0] # obtener la primera entrada

In [ ]:
df_I.values[1] # obtener la segunda entrada

In [ ]:
df_I.values[2:10] # obtener de la tercera a la décima entrada

In [ ]:
df_I.values[-1] # obtener la última entrada

In [ ]:
df_I.values[15::] # obtener todas las entradas a partir de la entrada 16

In [ ]:
df_I.values[-5::] # obtener las últimas cinco entradas

Con el slicing accederemos más adelante a los valores individuales y a los rangos de valores que queramos inspeccionar o graficar.

In [ ]:
np.max(df_I.values) # devuelve la entrada máxima

In [ ]:
1.1*np.max(df_I.values) # devuelve 1.1 veces la entrada máxima

Este último valor se eligió para dejar un poco de espacio entre el máximo de la línea graficada y el borde superior de la ventana del gráfico.

### Tarea 2: Calcular el número de vehículos eléctricos matriculados hasta ahora. Graficar y guardar el resultado.

Para cada cohorte de edad *c*, la serie temporal *I(c)* que leímos arriba indica el número total de vehículos nuevos matriculados. 

El parámetro _BEVshare(c,T)_ indica la proporción de vehículos de pasajeros convencionales (primera columna _T=0_) y eléctricos (segunda columna _T=1_) en las matriculaciones nuevas.

La división de la entrada total en los distintos tipos de tecnología, *I(c,T)*, está dada por la siguiente ecuación:

$$ I(c,T) = I(c) \cdot BEVshare(c,T) /100  $$

(¡Sí, se pueden insertar ecuaciones en formato \LaTeX en los campos de markdown de los cuadernos de Jupyter!)

El factor 1/100 se multiplica para convertir la unidad de valores porcentuales a proporción (unidad 1). Abajo, primero leemos los datos de *BEVshare(c,T)* al espacio de trabajo y luego calculamos la entrada desagregada, usando la función _np.einsum_ para multiplicar y sumar sobre múltiples índices.

In [ ]:
df_BEVshare = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='BEV_share', index_col=0)
df_BEVshare # mostrar los datos

In [ ]:
I_c          = df_I.values[:,0]   # convertir los valores del dataframe de pandas a un arreglo de numpy. Extraer solo la primera columna.
# Para series temporales, aquí debe extraerse la primera columna (índice: 0) para que los comandos siguientes funcionen.

BEV_share_cT = df_BEVshare.values # convertir los valores del dataframe de pandas a un arreglo de numpy

In [ ]:
I_c.shape # inspeccionar la forma de cada arreglo o serie temporal

In [ ]:
BEV_share_cT.shape # inspeccionar la forma de cada arreglo o serie temporal

In [ ]:
I_cT = np.einsum('c,cT->cT',I_c,BEV_share_cT[0:35,:]) / 100 
# Aquí pasamos dos arreglos, I_c y BEV_share_cT[0:35,:], a la función einsum.
# En la cadena de formato especificamos la forma de ambos: c y cT, y especificamos la forma del resultado final: ->cT.
# ¡Más sobre np.einsum en los siguientes cuadernos de software!

En el comando anterior se extraen las primeras 35 filas (cohortes de edad) para que el rango de cohortes de _BEVshare_ coincida con el de _I_, que cubre los años 1990-2024.

Graficamos el resultado:

In [ ]:
plt.plot(np.arange(1990,2025,1),I_cT) # graficar con los valores del eje x dados, generados con el comando np.arange
plt.ylim(0, 1.1*np.max(I_cT)) # fijar el máximo del eje y
plt.xlabel('cohorte de edad') # agregar etiqueta del eje x
plt.ylabel('matriculaciones nuevas (entrada), autos') # agregar etiqueta del eje y
plt.title('Entrada de autos de pasajeros por cohorte de edad y tecnología, 20 países de la zona euro') # agregar título
plt.legend(['vehículos a gasolina','vehículos eléctricos de batería']) # agregar una leyenda
plt.savefig('Inflow_EUR20_cT.png', dpi = 300) # guardar la figura con las modificaciones anteriores. ¡Sin el comando plt.show() cuando se guarda la figura!

Vemos que los vehículos eléctricos están en aumento en los países de la zona euro, con los siguientes valores y total:

In [ ]:
I_cT[:,1] # extraer la segunda columna (índice 1), posición de los BEV.

In [ ]:
I_cT[:,1].sum() # da la suma total de lo que esté antes del comando .sum().

La mayor matriculación de vehículos eléctricos hasta ahora en la región de estudio fue de aproximadamente 2.05 millones/año (en 2023), y el total acumulado hasta ahora fue de aproximadamente 9.1 millones de unidades.

**Exportar a Excel la serie temporal de entrada por tecnología**

Primero, creamos un nuevo dataframe con el resultado calculado y lo guardamos en Excel para archivarlo y para análisis/inspección posteriores.

In [ ]:
export_df= pd.DataFrame(data=I_cT,      # valores
            index=np.arange(1990,2025,1),        # 1ra columna como índice
            columns=['vehículos a gasolina','vehículos eléctricos de batería'])  # 1ra fila como nombres de las columnas
export_df

In [ ]:
export_df.to_excel('Bath_dMFA_2026_Software_1_Output_Single.xlsx', merge_cells=False)

Segundo, creamos otro dataframe y exportamos varios dataframes a diferentes hojas de Excel, usando una instancia de xlsxwriter.

In [ ]:
pd_xlsx_writer = pd.ExcelWriter('Bath_dMFA_2026_Software_1_Output_Multiple.xlsx')

In [ ]:
ExportData2 = np.cumsum(I_cT, axis = 0) # suma acumulada de toda la entrada hasta el año actual
ExportData2.shape

In [ ]:
export_df.to_excel(pd_xlsx_writer, sheet_name="Inflow_cT", merge_cells=False)   

export_df2= pd.DataFrame(data=ExportData2,      # valores: sumar I_cT sobre el primer eje (índice 0: cohorte de edad)
            index=np.arange(1990,2025,1),        # 1ra columna como índice
            columns=['vehículos a gasolina','vehículos eléctricos de batería'])   # 1ra fila como nombres de las columnas

export_df2.to_excel(pd_xlsx_writer, sheet_name="Inflow_Cumulative_T", merge_cells=False)   

pd_xlsx_writer.close()

Esta exportación concluye este ejercicio introductorio.